In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

# 查询数据
sql_1 = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_date,
    o.order_status,
    o.total_price_before_tax,
    oi.product_id,
    oi.quantity,
    oi.product_name,
    oi.unit_price,
    oi.line_price_before_tax,
    oi.is_free_gift,
    pa.campaign_id,
    p.product_name AS productinfo_product_name,
    p.product_m3_code,
    s.store_id,
    s.store_region,
    s.store_name
FROM "Order" o
JOIN "OrderItem" oi
    ON o.order_id = oi.order_id
JOIN "PromotionActivity" pa
    ON o.campaign_id = pa.campaign_id
JOIN "ProductInfo" p
    ON oi.product_id = p.product_id
JOIN "StoreInfo" s
    ON o.store_id = s.store_id
WHERE o.order_status IN ('Completed', 'Shipped');
"""

df_order_completed_shipped = pd.read_sql(sql_1, engine)

# 查看数据
df_order_completed_shipped

,order_id,customer_id,order_date,order_status,total_price_before_tax,product_id,quantity,product_name,unit_price,line_price_before_tax,is_free_gift,campaign_id,productinfo_product_name,product_m3_code,store_id,store_region,store_name
0,37,157800,2023-03-25 11:47:39,Completed,317.79,105,1,Ray-Ban Flacko Prescription,188.26,150.61,False,1,Ray-Ban Flacko Prescription,RB-FR-0105,17,Florida,Ray-Ban Tampa
1,37,157800,2023-03-25 11:47:39,Completed,317.79,97,1,Ray-Ban New Wayfarer Prescription,208.97,167.18,False,1,Ray-Ban New Wayfarer Prescription,RB-FR-0097,17,Florida,Ray-Ban Tampa
2,95,132132,2023-01-07 01:43:38,Completed,284.77,16,1,Ray-Ban Flacko Prescription,355.96,284.77,False,1,Ray-Ban Flacko Prescription,RB-FR-0016,31,Ohio,Ray-Ban Kenwood
3,109,76594,2023-02-22 10:47:17,Completed,1070.81,23,3,Ray-Ban Sam Prescription,435.29,1070.81,False,3,Ray-Ban Sam Prescription,RB-FR-0023,98,Illinois,LensCrafters 1055 W Bryn Mawr
4,133,81844,2023-02-09 00:08:23,Shipped,609.49,14,1,Ray-Ban Ray-Ban Reverse Prescription,281.05,224.84,False,1,Ray-Ban Ray-Ban Reverse Prescription,RB-FR-0014,8,California,Ray-Ban Abbot Kinney
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229638,499963,72391,2024-11-25 16:09:59,Shipped,615.53,42,1,Ray-Ban Ray-Ban Reverse Non-prescription,411.78,267.66,False,26,Ray-Ban Ray-Ban Reverse Non-prescription,RB-FR-0042,38,Washington,Ray-Ban Bellevue
229639,499987,136134,2024-10-25 05:50:04,Completed,744.64,152,1,Ray-Ban Ray-Ban Meta Prescription,254.33,206.01,False,22,Ray-Ban Ray-Ban Meta Prescription,RB-FR-0152,53,Florida,LensCrafters Colonial Drive
229640,499987,136134,2024-10-25 05:50:04,Completed,744.64,40,1,Ray-Ban Ray-Ban Meta Prescription,237.84,192.65,False,22,Ray-Ban Ray-Ban Meta Prescription,RB-FR-0040,53,Florida,LensCrafters Colonial Drive
229641,499987,136134,2024-10-25 05:50:04,Completed,744.64,69,1,Ray-Ban Round Metal Non-prescription,427.14,345.98,False,22,Ray-Ban Round Metal Non-prescription,RB-FR-0069,53,Florida,LensCrafters Colonial Drive


In [2]:
sql_2 = """
WITH
pa_meta AS (
    SELECT
        campaign_id,
        campaign_name,
        campaign_type,
        discount_type,
        start_date::date AS start_date,
        end_date::date AS end_date,
        (end_date::date - start_date::date + 1) AS campaign_days,
        expected_sales_lift
    FROM "PromotionActivity"
),

-- 用已打活动标签的订单反推活动覆盖范围：门店 + 商品
promo_scope AS (
    SELECT DISTINCT
        pa.campaign_id,
        o.store_id,
        oi.product_id
    FROM pa_meta pa
    JOIN "Order" o
      ON o.campaign_id = pa.campaign_id
     AND o.order_status IN ('Completed', 'Shipped')
     AND o.order_date::date BETWEEN pa.start_date AND pa.end_date
    JOIN "OrderItem" oi
      ON oi.order_id = o.order_id
),

-- 促销窗口：同店同商品范围内，活动期全部订单（排除其它活动污染）
promo_window_orders AS (
    SELECT
        pa.campaign_id,
        o.order_id,
        o.order_date::date AS order_date,
        o.campaign_id AS order_campaign_id,
        o.store_id,
        oi.product_id,
        oi.quantity,
        oi.unit_price,
        oi.line_price_before_tax,
        oi.is_free_gift
    FROM pa_meta pa
    JOIN promo_scope sc
      ON sc.campaign_id = pa.campaign_id
    JOIN "Order" o
      ON o.store_id = sc.store_id
     AND o.order_status IN ('Completed', 'Shipped')
     AND o.order_date::date BETWEEN pa.start_date AND pa.end_date
     AND (o.campaign_id IS NULL OR o.campaign_id = pa.campaign_id)
    JOIN "OrderItem" oi
      ON oi.order_id = o.order_id
     AND oi.product_id = sc.product_id
),

-- 基线窗口：活动前等长、同店同商品、仅非活动单
baseline_window_orders AS (
    SELECT
        pa.campaign_id,
        o.order_id,
        o.order_date::date AS order_date,
        o.store_id,
        oi.product_id,
        oi.quantity,
        oi.unit_price,
        oi.line_price_before_tax,
        oi.is_free_gift
    FROM pa_meta pa
    JOIN promo_scope sc
      ON sc.campaign_id = pa.campaign_id
    JOIN "Order" o
      ON o.store_id = sc.store_id
     AND o.order_status IN ('Completed', 'Shipped')
     AND o.campaign_id IS NULL
     AND o.order_date::date BETWEEN (pa.start_date - pa.campaign_days) AND (pa.start_date - 1)
    JOIN "OrderItem" oi
      ON oi.order_id = o.order_id
     AND oi.product_id = sc.product_id
),

promo_agg AS (
    SELECT
        p.campaign_id,
        COUNT(DISTINCT p.order_id) AS promo_order_count,
        SUM(CASE WHEN p.is_free_gift = FALSE OR p.is_free_gift IS NULL THEN p.line_price_before_tax ELSE 0 END) AS promo_sales_amount,
        SUM(CASE WHEN p.is_free_gift = FALSE OR p.is_free_gift IS NULL THEN p.quantity ELSE 0 END) AS promo_sales_qty,
        SUM(CASE WHEN p.is_free_gift = FALSE OR p.is_free_gift IS NULL THEN (p.line_price_before_tax - COALESCE(pi.cost_price,0)*p.quantity) ELSE 0 END) AS promo_gross_profit,
        -- 投入只统计活动标签单上的折扣，避免把自然折扣算进营销投入
        SUM(
            CASE WHEN p.order_campaign_id = p.campaign_id
                 THEN (p.unit_price * p.quantity) - p.line_price_before_tax
                 ELSE 0 END
        ) AS campaign_discount_total,
        SUM((p.unit_price * p.quantity) - p.line_price_before_tax) AS promo_window_discount_total
    FROM promo_window_orders p
    LEFT JOIN "ProductInfo" pi
      ON pi.product_id = p.product_id
    GROUP BY p.campaign_id
),

baseline_agg AS (
    SELECT
        b.campaign_id,
        COUNT(DISTINCT b.order_id) AS baseline_order_count,
        SUM(CASE WHEN b.is_free_gift = FALSE OR b.is_free_gift IS NULL THEN b.line_price_before_tax ELSE 0 END) AS baseline_sales_amount,
        SUM(CASE WHEN b.is_free_gift = FALSE OR b.is_free_gift IS NULL THEN b.quantity ELSE 0 END) AS baseline_sales_qty,
        SUM(CASE WHEN b.is_free_gift = FALSE OR b.is_free_gift IS NULL THEN (b.line_price_before_tax - COALESCE(pi.cost_price,0)*b.quantity) ELSE 0 END) AS baseline_gross_profit
    FROM baseline_window_orders b
    LEFT JOIN "ProductInfo" pi
      ON pi.product_id = b.product_id
    GROUP BY b.campaign_id
),

result_base AS (
    SELECT
        pa.campaign_id,
        pa.campaign_name,
        pa.campaign_type,
        pa.discount_type,
        pa.start_date,
        pa.end_date,
        pa.campaign_days,
        pa.expected_sales_lift,

        COALESCE(p.promo_order_count, 0) AS promo_order_count,
        COALESCE(p.promo_sales_amount, 0)::numeric AS promo_sales_amount,
        COALESCE(p.promo_sales_qty, 0) AS promo_sales_qty,
        COALESCE(p.promo_gross_profit, 0)::numeric AS promo_gross_profit,
        COALESCE(p.promo_sales_amount, 0)::numeric / NULLIF(COALESCE(p.promo_order_count,0),0) AS promo_aov,
        COALESCE(p.campaign_discount_total, 0)::numeric AS campaign_discount_total,

        COALESCE(b.baseline_order_count, 0) AS baseline_order_count,
        COALESCE(b.baseline_sales_amount, 0)::numeric AS baseline_sales_amount,
        COALESCE(b.baseline_sales_qty, 0) AS baseline_sales_qty,
        COALESCE(b.baseline_gross_profit, 0)::numeric AS baseline_gross_profit
    FROM pa_meta pa
    LEFT JOIN promo_agg p
      ON p.campaign_id = pa.campaign_id
    LEFT JOIN baseline_agg b
      ON b.campaign_id = pa.campaign_id
)

SELECT
    r.*,

    r.promo_sales_amount / NULLIF(r.campaign_days::numeric, 0) AS promo_daily_sales_amount,
    r.baseline_sales_amount / NULLIF(r.campaign_days::numeric, 0) AS baseline_daily_sales_amount,
    r.promo_gross_profit / NULLIF(r.campaign_days::numeric, 0) AS promo_daily_gross_profit,
    r.baseline_gross_profit / NULLIF(r.campaign_days::numeric, 0) AS baseline_daily_gross_profit,

    (r.promo_sales_amount - r.baseline_sales_amount) AS sales_lift,
    CASE WHEN r.baseline_sales_amount = 0 THEN NULL
         ELSE ROUND((r.promo_sales_amount - r.baseline_sales_amount) / r.baseline_sales_amount, 4)
    END AS sales_lift_rate,

    ((r.promo_sales_amount / NULLIF(r.campaign_days::numeric,0))
      - (r.baseline_sales_amount / NULLIF(r.campaign_days::numeric,0))) AS daily_sales_lift,
    CASE WHEN r.baseline_sales_amount = 0 THEN NULL
         ELSE ROUND(
              ((r.promo_sales_amount / NULLIF(r.campaign_days::numeric,0))
               - (r.baseline_sales_amount / NULLIF(r.campaign_days::numeric,0)))
              / (r.baseline_sales_amount / NULLIF(r.campaign_days::numeric,0)),
              4
         )
    END AS daily_sales_lift_rate,

    (r.promo_gross_profit - r.baseline_gross_profit) AS incremental_gross_profit,
    CASE WHEN r.campaign_discount_total = 0 THEN NULL
         ELSE ROUND((r.promo_gross_profit - r.baseline_gross_profit) / r.campaign_discount_total, 4)
    END AS roi_estimate,
    
    r.promo_gross_profit / NULLIF(r.promo_sales_amount, 0) AS promo_gross_margin,
    r.baseline_gross_profit / NULLIF(r.baseline_sales_amount, 0) AS baseline_gross_margin,
    ROUND(
      (r.promo_gross_profit / NULLIF(r.promo_sales_amount,0))
      - (r.baseline_gross_profit / NULLIF(r.baseline_sales_amount,0)),
      4
    ) AS incremental_gross_margin


FROM result_base r
ORDER BY r.campaign_id;
"""

df_promo_effect = pd.read_sql(sql_2, engine)
df_promo_effect["start_date"] = pd.to_datetime(df_promo_effect["start_date"])
df_promo_effect["end_date"] = pd.to_datetime(df_promo_effect["end_date"])

# 查看数据
df_promo_effect.to_parquet("2023-2024_promo_effect.parquet",engine='fastparquet',index=False)
print("已经保存到 2023-2024_promo_effect.parquet")
df_promo_effect


已经保存到 2023-2024_promo_effect.parquet


,campaign_id,campaign_name,campaign_type,discount_type,start_date,end_date,campaign_days,expected_sales_lift,promo_order_count,promo_sales_amount,...,baseline_daily_gross_profit,sales_lift,sales_lift_rate,daily_sales_lift,daily_sales_lift_rate,incremental_gross_profit,roi_estimate,promo_gross_margin,baseline_gross_margin,incremental_gross_margin
0,1,2023 New Year Spectacle Sale,Holiday,Percentage,2023-01-01,2023-01-31,31,1.12,2668,961841.77,...,0.000000,961841.77,NaN,31027.153871,NaN,485619.96,4.6848,0.504885,NaN,NaN
1,2,2023 Valentine’s Day Lens Event,Holiday,Fixed Amount,2023-02-01,2023-02-28,28,1.18,1274,538176.42,...,5492.020357,258854.17,0.9267,9244.791786,0.9267,135534.89,7.8344,0.537577,0.550535,-0.0130
2,3,2023 Spring Sale,Seasonal,Percentage,2023-03-01,2023-04-30,61,1.10,12333,3634692.86,...,22448.781803,1165027.69,0.4717,19098.814590,0.4717,547285.33,2.2908,0.527324,0.554478,-0.0272
3,4,2023 Easter Eye Care Gift,Member,Free Gift,2023-04-01,2023-04-30,30,1.08,627,298250.63,...,2604.672000,158850.27,1.1395,5295.009000,1.1395,88178.62,29.2922,0.557648,0.560545,-0.0029
4,5,2023 Mother’s Day Special,Holiday,Percentage,2023-05-01,2023-05-31,31,1.16,1512,559002.65,...,5591.424516,245462.42,0.7829,7918.142581,0.7829,108492.67,1.8196,0.504160,0.552829,-0.0487
5,6,2023 Summer Sunglass Sale,Seasonal,Percentage,2023-06-01,2023-06-30,30,1.09,3276,966185.94,...,10698.696333,384124.76,0.6599,12804.158667,0.6599,182144.07,2.4068,0.520712,0.551421,-0.0307
6,7,2023 July 4 Independence Event,Holiday,Fixed Amount,2023-07-01,2023-07-31,31,1.13,2552,1059716.55,...,8289.885484,595354.94,1.2821,19204.998065,1.2821,311605.35,7.0182,0.536551,0.553419,-0.0169
7,8,2023 August Summer Clearance,Seasonal,Percentage,2023-08-01,2023-08-31,31,1.11,9589,2718512.43,...,30731.850000,1010999.56,0.5921,32612.889032,0.5921,452048.67,1.7470,0.516730,0.557939,-0.0412
8,9,2023 Back to School,Seasonal,Buy One Get One,2023-09-01,2023-09-30,30,1.20,3853,1565861.73,...,16446.962333,677337.75,0.7623,22577.925000,0.7623,376099.24,1.0260,0.555290,0.555313,0.0000
9,10,2023 Halloween Frame Treat,Holiday,Percentage,2023-10-01,2023-10-31,31,1.07,15045,5259150.91,...,43014.464839,2866510.86,1.1981,92468.092258,1.1981,1437238.93,3.5136,0.526832,0.557313,-0.0305


In [3]:
sql_3 = """
select * from "PromotionActivity"
"""
df_promotion_activity = pd.read_sql(sql_3, engine)

# 查看数据
df_promotion_activity.columns

Index(['campaign_id', 'campaign_type', 'campaign_name', 'target_audience',
       'region', 'start_date', 'end_date', 'discount_type', 'discount_value',
       'min_purchase_amount', 'expected_sales_lift', 'status'],
      dtype='object')

In [3]:
df_promo_effect.columns

Index(['campaign_id', 'campaign_name', 'campaign_type', 'discount_type',
       'start_date', 'end_date', 'campaign_days', 'expected_sales_lift',
       'promo_order_count', 'promo_sales_amount', 'promo_sales_qty',
       'promo_gross_profit', 'promo_aov', 'campaign_discount_total',
       'baseline_order_count', 'baseline_sales_amount', 'baseline_sales_qty',
       'baseline_gross_profit', 'promo_daily_sales_amount',
       'baseline_daily_sales_amount', 'promo_daily_gross_profit',
       'baseline_daily_gross_profit', 'sales_lift', 'sales_lift_rate',
       'daily_sales_lift', 'daily_sales_lift_rate', 'incremental_gross_profit',
       'roi_estimate', 'promo_gross_margin', 'baseline_gross_margin',
       'incremental_gross_margin'],
      dtype='object')